<a href="https://colab.research.google.com/github/abdull254/wikiart-museum-style/blob/main/wikiart_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WikiArt to Museum: training on Colab

Before you start: **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

Your data must be at `My Drive/colab_data.zip` (made by `pack_for_colab.py`).

## 1. Check the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Connect Google Drive and unpack the data
Copying the zip to Colab's own disk first makes training much faster than reading from Drive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/colab_data.zip /content/
!cd /content && unzip -q -o colab_data.zip && rm colab_data.zip
!ls /content && ls /content/data/final

Mounted at /content/drive
data  drive  sample_data  train.py
final_styles.csv  museum_table.csv  wikiart_split.csv
hand_check.csv	  museum_test.csv   wikiart_table.csv


## 3. Quick test (about 1–2 minutes)

In [3]:
%cd /content
!python train.py --quick --workers 2

/content
Device: cuda   classes: 6   train 2000  val 500  test 500
  Baroque                           337 train images
  Mannerism (Late Renaissance)      108 train images
  Northern Renaissance              130 train images
  Realism                           615 train images
  Rococo                            230 train images
  Romanticism                       580 train images
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 175MB/s]
  loading the first batch of images...
  batch 1 loaded after 0.7 s, training on it...
  batch 1 done after 4.8 s (loss 1.910)
  batch 2 loaded after 4.8 s, training on it...
  batch 2 done after 5.1 s (loss 1.943)
  batch 3 loaded after 5.1 s, training on it...
  batch 3 done after 5.3 s (loss 1.886)
  speed: 0.70 s per batch -> about 3 min per full epoch, 0.7 h for a 12-epoch run
epoch 1: train loss 1.836 | val acc 0.202 bal acc 0.255 m

## 4. Full run
Change `SEED` for each repeat (1, 2, 3...). Each run is copied to `My Drive/wikiart_runs/` when it finishes, so nothing is lost if Colab disconnects.

In [ ]:
SEED = 3
%cd /content
!python train.py --seed {SEED} --eval-museums --workers 2

# Copy to Drive ONLY if the run finished (metrics.json exists and best.pt is real),
# and never overwrite an existing run on Drive.
import os, shutil, time
src = f'/content/runs/resnet50_seed{SEED}'
dst = f'/content/drive/MyDrive/wikiart_runs/resnet50_seed{SEED}'
ok = os.path.exists(f'{src}/metrics.json') and os.path.getsize(f'{src}/best.pt') > 10_000_000
if not ok:
    print('Run did NOT finish (no metrics.json or empty best.pt). Nothing copied to Drive.')
else:
    if os.path.exists(dst):
        old = dst + time.strftime('_old_%Y%m%d_%H%M%S')
        os.rename(dst, old)
        print('Existing Drive run moved to', old)
    shutil.copytree(src, dst)
    size = os.path.getsize(f'{dst}/best.pt')
    print(f'Saved to Drive, best.pt = {size/1e6:.1f} MB')
    assert size > 10_000_000, 'best.pt on Drive is empty: check Drive storage space'

/content
Device: cuda   classes: 6   train 18501  val 3786  test 3857
  Baroque                          3003 train images
  Mannerism (Late Renaissance)      920 train images
  Northern Renaissance             1252 train images
  Realism                          6055 train images
  Rococo                           1759 train images
  Romanticism                      5512 train images
  loading the first batch of images...
  batch 1 loaded after 0.5 s, training on it...
  batch 1 done after 2.1 s (loss 1.909)
  batch 2 loaded after 2.1 s, training on it...
  batch 2 done after 2.4 s (loss 1.938)
  batch 3 loaded after 2.4 s, training on it...
  batch 3 done after 2.6 s (loss 1.928)
  speed: 0.43 s per batch -> about 2 min per full epoch, 0.4 h for a 12-epoch run
  epoch 1 batch 50/290  loss 1.907
  epoch 1 batch 100/290  loss 1.896
  epoch 1 batch 150/290  loss 1.858
  epoch 1 batch 200/290  loss 1.793
  epoch 1 batch 250/290  loss 1.729
epoch 1: train loss 1.681 | val acc 0.398 bal ac

## 5. Results

In [ ]:
import json
m = json.load(open(f'/content/runs/resnet50_seed{SEED}/metrics.json'))
for k, v in m.items():
    print(f"{k:<14} n={v['n']:>5}  bal acc {v['balanced_accuracy']:.3f}  macro-F1 {v['macro_f1']:.3f}  ECE {v['ece']:.3f}")
    for g in ('seen_artists', 'unseen_artists'):
        if g in v:
            print(f"    {g:<15} n={v[g]['n']:>5}  bal acc {v[g]['balanced_accuracy']:.3f}")

## 6. Test-time ablations (no retraining, about 5 minutes per model)
Scores a trained model on changed museum images: crop, 512 px, colour-normalized, JPEG-compressed and grayscale. Run once per seed you have trained.

In [ ]:
SEED = 3
!cp /content/drive/MyDrive/evaluate.py /content/
%cd /content
import os, json, shutil
run = f'/content/runs/resnet50_seed{SEED}'
drive_run = f'/content/drive/MyDrive/wikiart_runs/resnet50_seed{SEED}'
# Always start from the finished run saved on Drive, never a leftover local folder
# (a local folder may be from an interrupted run).
if os.path.exists(run):
    shutil.rmtree(run)
shutil.copytree(drive_run, run)
assert os.path.exists(f'{run}/metrics.json'), 'No metrics.json: this run never finished'
assert os.path.getsize(f'{run}/best.pt') > 10_000_000, 'best.pt is empty or partial: retrain this seed'
expected = json.load(open(f'{run}/metrics.json'))['wikiart_test']['balanced_accuracy']
print(f'Training-time WikiArt test balanced accuracy: {expected:.3f}')
!python evaluate.py --run runs/resnet50_seed{SEED}
# Sanity check: the "base wikiart_test" row above must match the number printed before it (within 0.01).
# If it does not, the checkpoint is not the model that produced metrics.json, and the ablations are invalid.
!cp {run}/ablations.csv {run}/preds_*_*.csv {drive_run}/
print('Saved ablations to Drive')

## 7. Grayscale-training ablation (retrains, about 20 minutes)
Trains with 20% of images turned grayscale, to test whether relying on colour explains the museum gap.

In [ ]:
SEED = 1
%cd /content
!python train.py --seed {SEED} --grayscale-aug 0.2 --eval-museums --workers 2
!cp -r /content/runs/resnet50_gray0.2_seed{SEED} /content/drive/MyDrive/wikiart_runs/
print('Saved to Drive')

## 8. Statistics over seeds
First run the check: every seed must show a best.pt of about 94 MB. Then run the statistics cell.

In [ ]:
# Check every run before computing statistics: a real checkpoint is about 94 MB.
!ls -la /content/drive/MyDrive/wikiart_runs/*/best.pt

In [ ]:
!cp /content/drive/MyDrive/stats.py /content/
R = '/content/drive/MyDrive/wikiart_runs'
!python /content/stats.py --runs {R}/resnet50_seed1 {R}/resnet50_seed2 {R}/resnet50_seed3

In [ ]:
# (removed: deleting local runs is not needed)